In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
from entsoe import EntsoePandasClient

In [18]:
Year = 2024
Save_DIR = f'./Entso-e_data/{Year}'
import os

# Create the directory if it doesn't exist
os.makedirs(Save_DIR, exist_ok=True)

In [19]:
def load_client() -> EntsoePandasClient:
    load_dotenv()


    api_key = os.getenv("ENTSOE_API_KEY")
    if not api_key:
        raise ValueError("Missing ENTSOE_API_KEY in .env")

    return EntsoePandasClient(api_key=api_key)

In [20]:

client = load_client()

# start = pd.Timestamp('20171201', tz='Europe/Brussels')
# end = pd.Timestamp('20180101', tz='Europe/Brussels')
# country_code = 'IE'  # Belgium
# country_code_from = 'FR'  # France
# country_code_to = 'DE_LU' # Germany-Luxembourg
# type_marketagreement_type = 'A01'
# contract_marketagreement_type = "A01"
# process_type = 'A51'


In [21]:
start = pd.Timestamp(f'{Year}0101', tz='Europe/Brussels')
end = pd.Timestamp(f'{Year+1}0101', tz='Europe/Brussels')
country_code = 'IE'  # Ireland
wind_solar_forecast = client.query_wind_and_solar_forecast(country_code, start=start, end=end, psr_type=None)

In [22]:
wind_solar_forecast

,Wind Onshore
2023-12-31 23:00:00+00:00,3099.500
2024-01-01 00:00:00+00:00,2777.415
2024-01-01 01:00:00+00:00,2525.490
2024-01-01 02:00:00+00:00,2250.055
2024-01-01 03:00:00+00:00,1900.400
...,...
2024-12-31 18:00:00+00:00,3158.595
2024-12-31 19:00:00+00:00,2978.725
2024-12-31 20:00:00+00:00,2896.355
2024-12-31 21:00:00+00:00,2772.555


In [23]:
# wind_solar_intraday_forecast = client.query_intraday_wind_and_solar_forecast(country_code, start=start, end=end, psr_type=None)


In [24]:
actual_generation = client.query_generation(country_code, start=start, end=end, psr_type=None)
# If your df has MultiIndex columns (two header rows)
# actual_generation.columns = actual_generation.columns.get_level_values(0)

In [25]:
if isinstance(actual_generation.columns, pd.MultiIndex):
    actual_generation.columns = [
        f"{lvl0} [{lvl1}]"
        for lvl0, lvl1 in actual_generation.columns
    ]
actual_generation

,Fossil Gas [Actual Aggregated],Fossil Hard coal [Actual Aggregated],Fossil Oil [Actual Aggregated],Fossil Peat [Actual Aggregated],Hydro Pumped Storage [Actual Aggregated],Hydro Run-of-river and pondage [Actual Aggregated],Wind Onshore [Actual Aggregated],Other [Actual Aggregated],Fossil Gas [Actual Consumption],Fossil Hard coal [Actual Consumption],Fossil Oil [Actual Consumption],Fossil Peat [Actual Consumption],Hydro Pumped Storage [Actual Consumption],Hydro Run-of-river and pondage [Actual Consumption],Other [Actual Consumption],Solar [Actual Aggregated],Solar [Actual Consumption]
2023-12-31 23:00:00+00:00,402.12,114.10,0.54,42.29,0.00,180.76,2514.23,0.0,4.58,6.64,3.10,0.0,72.15,0.0,0.66,NaN,NaN
2023-12-31 23:30:00+00:00,365.44,104.05,0.54,42.43,0.00,178.56,2297.56,0.0,4.58,6.87,2.80,0.0,73.03,0.0,0.67,NaN,NaN
2024-01-01 00:00:00+00:00,356.76,86.89,0.45,42.22,0.00,178.16,2262.78,0.0,4.58,6.87,2.52,0.0,73.03,0.0,0.66,NaN,NaN
2024-01-01 00:30:00+00:00,409.18,104.96,0.45,43.34,0.00,177.69,2228.26,0.0,4.58,6.87,2.52,0.0,71.78,0.0,0.66,NaN,NaN
2024-01-01 01:00:00+00:00,406.54,104.73,0.45,42.43,0.00,177.28,2128.10,0.0,6.18,6.87,3.19,0.0,71.78,0.0,0.67,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 20:30:00+00:00,310.92,0.00,164.37,41.06,19.77,111.01,2188.83,0.0,1.39,8.48,0.05,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 21:00:00+00:00,326.58,0.00,171.43,41.20,22.70,93.77,2114.85,0.0,2.06,8.93,0.07,0.0,0.00,0.0,0.67,0.0,0.0
2024-12-31 21:30:00+00:00,310.79,0.00,163.03,41.06,130.01,92.85,2070.20,0.0,2.08,8.93,0.07,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 22:00:00+00:00,314.80,0.00,164.37,41.20,19.84,92.27,2002.77,0.0,2.33,8.48,0.07,0.0,0.00,0.0,0.65,0.0,0.0


In [26]:
load = client.query_load(country_code, start=start, end=end)

In [27]:

load_forecast = client.query_load_forecast(country_code, start=start, end=end)

In [28]:
wind_solar_forecast = wind_solar_forecast.rename(
    columns={col: f"{col} [forecast-entsoe]" for col in wind_solar_forecast.columns}
)
load_forecast = load_forecast.rename(
    columns={col: f"{col} [forecast-entsoe]" for col in load_forecast.columns}
)
actual_generation = actual_generation.rename(
    columns={col: f"{col} [actual-entsoe]" for col in actual_generation.columns}
)
load = load.rename(
    columns={col: f"{col} [actual-entsoe]" for col in load.columns}   )

In [29]:
actual_generation

,Fossil Gas [Actual Aggregated] [actual-entsoe],Fossil Hard coal [Actual Aggregated] [actual-entsoe],Fossil Oil [Actual Aggregated] [actual-entsoe],Fossil Peat [Actual Aggregated] [actual-entsoe],Hydro Pumped Storage [Actual Aggregated] [actual-entsoe],Hydro Run-of-river and pondage [Actual Aggregated] [actual-entsoe],Wind Onshore [Actual Aggregated] [actual-entsoe],Other [Actual Aggregated] [actual-entsoe],Fossil Gas [Actual Consumption] [actual-entsoe],Fossil Hard coal [Actual Consumption] [actual-entsoe],Fossil Oil [Actual Consumption] [actual-entsoe],Fossil Peat [Actual Consumption] [actual-entsoe],Hydro Pumped Storage [Actual Consumption] [actual-entsoe],Hydro Run-of-river and pondage [Actual Consumption] [actual-entsoe],Other [Actual Consumption] [actual-entsoe],Solar [Actual Aggregated] [actual-entsoe],Solar [Actual Consumption] [actual-entsoe]
2023-12-31 23:00:00+00:00,402.12,114.10,0.54,42.29,0.00,180.76,2514.23,0.0,4.58,6.64,3.10,0.0,72.15,0.0,0.66,NaN,NaN
2023-12-31 23:30:00+00:00,365.44,104.05,0.54,42.43,0.00,178.56,2297.56,0.0,4.58,6.87,2.80,0.0,73.03,0.0,0.67,NaN,NaN
2024-01-01 00:00:00+00:00,356.76,86.89,0.45,42.22,0.00,178.16,2262.78,0.0,4.58,6.87,2.52,0.0,73.03,0.0,0.66,NaN,NaN
2024-01-01 00:30:00+00:00,409.18,104.96,0.45,43.34,0.00,177.69,2228.26,0.0,4.58,6.87,2.52,0.0,71.78,0.0,0.66,NaN,NaN
2024-01-01 01:00:00+00:00,406.54,104.73,0.45,42.43,0.00,177.28,2128.10,0.0,6.18,6.87,3.19,0.0,71.78,0.0,0.67,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 20:30:00+00:00,310.92,0.00,164.37,41.06,19.77,111.01,2188.83,0.0,1.39,8.48,0.05,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 21:00:00+00:00,326.58,0.00,171.43,41.20,22.70,93.77,2114.85,0.0,2.06,8.93,0.07,0.0,0.00,0.0,0.67,0.0,0.0
2024-12-31 21:30:00+00:00,310.79,0.00,163.03,41.06,130.01,92.85,2070.20,0.0,2.08,8.93,0.07,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 22:00:00+00:00,314.80,0.00,164.37,41.20,19.84,92.27,2002.77,0.0,2.33,8.48,0.07,0.0,0.00,0.0,0.65,0.0,0.0


In [30]:


actual_generation.to_csv(f"{Save_DIR}/actual_generation_ENTSOE.csv", index=True, index_label="DateTime")

In [31]:
wind_solar_forecast.to_csv(f"{Save_DIR}/wind_solar_forecast_ENTSOE.csv", index=True, index_label="DateTime")

In [32]:
load.to_csv(f"{Save_DIR}/load_ENTSOE.csv", index=True, index_label="DateTime")
load_forecast.to_csv(f"{Save_DIR}/load_forecast_ENTSOE.csv", index=True, index_label="DateTime")

In [33]:
actual_generation

,Fossil Gas [Actual Aggregated] [actual-entsoe],Fossil Hard coal [Actual Aggregated] [actual-entsoe],Fossil Oil [Actual Aggregated] [actual-entsoe],Fossil Peat [Actual Aggregated] [actual-entsoe],Hydro Pumped Storage [Actual Aggregated] [actual-entsoe],Hydro Run-of-river and pondage [Actual Aggregated] [actual-entsoe],Wind Onshore [Actual Aggregated] [actual-entsoe],Other [Actual Aggregated] [actual-entsoe],Fossil Gas [Actual Consumption] [actual-entsoe],Fossil Hard coal [Actual Consumption] [actual-entsoe],Fossil Oil [Actual Consumption] [actual-entsoe],Fossil Peat [Actual Consumption] [actual-entsoe],Hydro Pumped Storage [Actual Consumption] [actual-entsoe],Hydro Run-of-river and pondage [Actual Consumption] [actual-entsoe],Other [Actual Consumption] [actual-entsoe],Solar [Actual Aggregated] [actual-entsoe],Solar [Actual Consumption] [actual-entsoe]
2023-12-31 23:00:00+00:00,402.12,114.10,0.54,42.29,0.00,180.76,2514.23,0.0,4.58,6.64,3.10,0.0,72.15,0.0,0.66,NaN,NaN
2023-12-31 23:30:00+00:00,365.44,104.05,0.54,42.43,0.00,178.56,2297.56,0.0,4.58,6.87,2.80,0.0,73.03,0.0,0.67,NaN,NaN
2024-01-01 00:00:00+00:00,356.76,86.89,0.45,42.22,0.00,178.16,2262.78,0.0,4.58,6.87,2.52,0.0,73.03,0.0,0.66,NaN,NaN
2024-01-01 00:30:00+00:00,409.18,104.96,0.45,43.34,0.00,177.69,2228.26,0.0,4.58,6.87,2.52,0.0,71.78,0.0,0.66,NaN,NaN
2024-01-01 01:00:00+00:00,406.54,104.73,0.45,42.43,0.00,177.28,2128.10,0.0,6.18,6.87,3.19,0.0,71.78,0.0,0.67,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 20:30:00+00:00,310.92,0.00,164.37,41.06,19.77,111.01,2188.83,0.0,1.39,8.48,0.05,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 21:00:00+00:00,326.58,0.00,171.43,41.20,22.70,93.77,2114.85,0.0,2.06,8.93,0.07,0.0,0.00,0.0,0.67,0.0,0.0
2024-12-31 21:30:00+00:00,310.79,0.00,163.03,41.06,130.01,92.85,2070.20,0.0,2.08,8.93,0.07,0.0,0.00,0.0,0.66,0.0,0.0
2024-12-31 22:00:00+00:00,314.80,0.00,164.37,41.20,19.84,92.27,2002.77,0.0,2.33,8.48,0.07,0.0,0.00,0.0,0.65,0.0,0.0
